In [0]:
%run ./transformation_nb

In [0]:
%run ./data_quality_nb

In [0]:
from pyspark.sql.functions import current_timestamp


# ---------------------------
# Step 1: Load bronze dataset
# ---------------------------
bronze_df = spark.table("oil_gas_mlops.bronze.bronze_supply_chain")

# ---------------------------
# Step 2: Drop unwanted columns
# ---------------------------
columns_to_drop = ["ingestion_timestamp", "source_system", "operation_type"]
bronze_df = bronze_df.drop(*columns_to_drop)

# ---------------------------
# Step 3: Call transformation notebook (type casting)
# ---------------------------
silver_df = run_transformation(
    spark=spark,
    source_df=bronze_df,
    config_table="oil_gas_mlops.config.config_transformation",
    source_table_name="bronze_supply_chain"
)

# ---------------------------
# Step 4: Call DQ checks notebook
# ---------------------------
clean_df = run_dq_checks(
    spark=spark,
    source_df=silver_df,
    config_table="oil_gas_mlops.config.config_dq",
    source_table_name="bronze_supply_chain"
)

# ---------------------------
# Step 5: Add silver-layer timestamp (optional, tracks when silver processing happened)
# ---------------------------
clean_df = clean_df.withColumn("silver_processed_ts", current_timestamp())

# ---------------------------
# Step 6: Write to silver schema
# ---------------------------
clean_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("oil_gas_mlops.silver.silver_supply_chain")

print("Silver table created: oil_gas_mlops.silver.silver_supply_chain")